# FAQ Vector Search — Full Pipeline

This notebook is the **master guide** for the project. It runs the complete pipeline
from end to end and explains how every file in the repo fits together.

Run cells top-to-bottom once; the shared objects produced here (`model`, `documents`,
`vectors`, `X`) are exactly what the focused notebooks depend on.

---

## Repository file map (sequential order)

| # | File / folder | Purpose |
|---|---------------|---------|
| 0 | `pyproject.toml` / `.python-version` | Project metadata and locked Python version |
| 1 | `ingest.py` | Download FAQ data from DataTalks.Club; build minsearch keyword index |
| 2 | `embeddings.ipynb` | **Core exploration**: model loading, dot-product similarity, batch encoding, NumPy top-k, minsearch VectorSearch, SQLite VectorSearchIndex, RAG demos |
| 3 | `minsearch-vector.ipynb` | Focused look at `minsearch.VectorSearch` — **needs `X` and `documents` from Stage 2** |
| 4 | `rag_helper.py` | `RAGBase` class: search → context → prompt → LLM stream |
| 5 | `rag-vector.ipynb` | RAG pipeline wired to a `VectorSearch` index — **needs `X` and `documents` from Stage 2** |
| 6 | `sqlitesearch-vector.ipynb` | Open the persisted `faq_vectors2.db` and run SQLite-backed RAG |
| 7 | `pgvector-vector.ipynb` | PostgreSQL + pgvector backend; HNSW index; pgvector-backed RAG |
| 8 | `main.py` | CLI pipeline: ingest → embed → vector search → optional RAG |
| 9 | `orchestration/` | Kestra workflow orchestration layer (docker-compose + YAML flows + lesson notes) |

---

## Dependency graph

```
ingest.py
    └── documents, index
            │
            ▼
    embeddings.ipynb  ──────────────────────────────────────────┐
    (model, vectors, X)                                         │
            │                                                   │
     ┌──────┴──────────────────────┐                           │
     ▼                             ▼                           │
minsearch-vector.ipynb    sqlitesearch-vector.ipynb            │
  (needs X, documents)      (needs faq_vectors2.db)            │
     │                                                         │
     ▼                                                         │
rag_helper.py  ◄───────────────────────────────────────────────┘
(RAGBase)                    pgvector-vector.ipynb
     │                         (standalone, needs Postgres)
     ▼
rag-vector.ipynb
  (needs X, documents)
```

The orchestration layer (`orchestration/`) is independent — it uses Kestra to orchestrate
similar RAG and agent patterns via YAML flows rather than Python notebooks.


---
## Stage 1 — Data Ingestion
**File:** [`ingest.py`](ingest.py)

`ingest.py` exposes two helpers:
- `load_faq_data()` — fetches all DataTalks.Club course FAQs and returns a flat list of dicts with `question`, `answer`, `section`, `course` fields.
- `build_index(documents)` — wraps `minsearch.Index` for lightweight keyword search over those fields.

Everything else in the project starts here.

In [ ]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

print(f"Loaded {len(documents)} FAQ documents")
print("Sample document:", documents[0])

---
## Stage 2 — Embeddings
**File:** [`embeddings.ipynb`](embeddings.ipynb) ← open this for the full exploration

This stage:
1. Loads `all-MiniLM-L6-v2` from `sentence-transformers`.
2. Shows dot-product similarity between individual question/answer vectors.
3. Encodes all FAQ documents in batches → stores them in a NumPy array `X`.
4. Demonstrates manual top-k retrieval with `np.argsort`.

The objects `model`, `vectors`, and `X` produced here are shared by Stages 3, 5, and 6.

In [ ]:
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
import numpy as np

# Shared embedding model — reused across all vector-search backends.
model = SentenceTransformer("all-MiniLM-L6-v2")

# Combine question + answer into one text per document for richer embeddings.
texts = [doc["question"] + "    " + doc["answer"] for doc in documents]

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i: i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

# 2-D NumPy array: (num_docs, embedding_dim)
X = np.array(vectors)
print(f"Embedding matrix shape: {X.shape}")

In [ ]:
# Quick manual similarity search to verify the embeddings work.
query = "Can I still join the course after the start date?"
v_query = model.encode(query)
scores = X.dot(v_query)

top5 = np.argsort(scores)[-5:][::-1]
print(f"Top-5 matches for: '{query}'\n")
for idx in top5:
    print(f"  [{scores[idx]:.4f}] {documents[idx]['question']}")

---
## Stage 3 — In-Memory Vector Search with minsearch
**File:** [`minsearch-vector.ipynb`](minsearch-vector.ipynb) ← open for the focused demo

`minsearch.VectorSearch` wraps the numpy dot-product search in a clean API with
optional keyword filtering. It is **in-memory only** — no persistence.

> **Note:** `minsearch-vector.ipynb` expects `X` and `documents` to already be defined
> (it is meant to be run after Stage 2). Run the cells above first, or run this
> notebook which defines them.

In [ ]:
from minsearch import VectorSearch

# Fit the in-memory vector index over the pre-computed embeddings.
vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

# Unfiltered search.
results = vindex.search(query_vector, num_results=5)
print("Unfiltered top result:", results[0]["question"])

# Course-filtered search.
filtered = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)
print("Filtered top result:", filtered[0]["question"])

---
## Stage 4 — RAG Helper
**File:** [`rag_helper.py`](rag_helper.py)

`rag_helper.py` defines `RAGBase`, a reusable class that wires together:
- `search(query)` — delegates to whichever index is injected
- `build_context(results)` — formats retrieved docs into a prompt block
- `build_prompt(query, results)` — inserts question + context into the template
- `llm(prompt)` — streams the model response and returns the full text
- `rag(query)` — end-to-end: retrieve → prompt → generate

All specialised subclasses in the notebooks (`RAGVector`, `RAGPgVector`) extend this
base and only override `search()`.

In [ ]:
# Preview the RAGBase interface (no LLM client needed to inspect).
import inspect
from rag_helper import RAGBase

for name, method in inspect.getmembers(RAGBase, predicate=inspect.isfunction):
    sig = inspect.signature(method)
    print(f"  RAGBase.{name}{sig}")

---
## Stage 5 — RAG with Vector Search
**File:** [`rag-vector.ipynb`](rag-vector.ipynb) ← open for the focused demo

This notebook subclasses `RAGBase` to replace keyword `search()` with vector search
using the `minsearch.VectorSearch` index built above.

> **Note:** `rag-vector.ipynb` references `X` and `documents` without redefining them.
> Those variables come from Stage 2. Run the shared setup cells above, or use this
> notebook — both work.

An LLM client (Ollama or OpenAI-compatible) is required to call `.rag()`. Configure
the credentials in a `.env` file (see the `OLLAMA_API_KEY` / `OPENAI_API_KEY` vars
already referenced in the notebooks).

In [ ]:
from rag_helper import RAGBase


class RAGVector(RAGBase):
    """RAGBase subclass that uses a VectorSearch index instead of keyword search."""

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        return self.index.search(
            query_vector,
            filter_dict={"course": self.course},
            num_results=num_results,
        )


# ── LLM client setup (requires .env with OLLAMA_API_KEY) ──────────────────────
# Uncomment and run once your credentials are in place.
#
# from dotenv import load_dotenv
# import os
# from ollama import Client
#
# load_dotenv()
# client = Client(
#     host="https://ollama.com",
#     headers={"Authorization": "Bearer " + os.environ["OLLAMA_API_KEY"]},
# )
#
# vector_assistant = RAGVector(
#     embedder=model,
#     index=vindex,
#     llm_client=client,
# )
# vector_assistant.rag("the program has already begun, can I still sign up?")

print("RAGVector class defined. Uncomment the block above to run with a live LLM.")

---
## Stage 6 — Persistent SQLite Vector Search
**File:** [`sqlitesearch-vector.ipynb`](sqlitesearch-vector.ipynb) ← open for the focused demo

`sqlitesearch.VectorSearchIndex` persists the embeddings to a SQLite database file
(`faq_vectors2.db`) so they can be reloaded across Python sessions without
re-running the expensive batch encoding step.

Two sub-stages:
1. **Build** — call `.fit(vectors, documents)` once to write to disk.
2. **Reload** — open the same `db_path` in a new session and search immediately.

> `sqlitesearch-vector.ipynb` assumes the DB already exists (built by Stage 2 /
> `embeddings.ipynb`). The cell below covers both cases.

In [ ]:
import os
from sqlitesearch import VectorSearchIndex

DB_PATH = "faq_vectors2.db"

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path=DB_PATH,
)

if not os.path.exists(DB_PATH) or os.path.getsize(DB_PATH) < 1024:
    print("Building SQLite index for the first time...")
    vs_index.fit(vectors, documents)
    print("Index saved to", DB_PATH)
else:
    print(f"Reusing existing index at {DB_PATH}")

query_vector = model.encode("I just discovered the course. Can I still join it?")
results = vs_index.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=2,
)

for r in results:
    print(f"  Q: {r['question']}")

vs_index.close()

---
## Stage 7 — PostgreSQL Vector Search
**File:** [`pgvector-vector.ipynb`](pgvector-vector.ipynb) ← open for the focused demo

This is the only backend that requires external infrastructure. It uses the
`pgvector` Postgres extension to store and query dense vectors with a native HNSW
index.

### Prerequisites

Start a Postgres instance with `pgvector` enabled. The simplest way is:

```bash
docker run -e POSTGRES_PASSWORD=pass -e POSTGRES_DB=faq \
  -p 5432:5432 ankane/pgvector
```

Then update the connection string in the notebook before running it.

### What the notebook covers

1. Re-encodes all FAQs (self-contained, no shared state needed).
2. Creates a `documents` table with an `embedding vector(384)` column.
3. Inserts all rows with their vectors.
4. Creates an HNSW cosine index for fast ANN search.
5. Defines `pgvector_search()` and `RAGPgVector` (extends `RAGBase`).
6. Runs a full RAG query.

> The connection string in `pgvector-vector.ipynb` contains a placeholder (`******`).
> Replace it with your actual credentials before running.

---
## Stage 8 — CLI Entry Point
**File:** [`main.py`](main.py)

`main.py` is the `.py` equivalent of this notebook — the same five stages wired
into a command-line script you can run without Jupyter.

```bash
# Stage 1-4: vector search only (no LLM needed)
uv run python main.py --query "Can I still join after the start date?"

# Stage 1-5: full RAG answer (needs OLLAMA_API_KEY or OPENAI_API_KEY in .env)
uv run python main.py --query "Can I still join after the start date?" --rag

# Filter to one course, show 10 results
uv run python main.py --query "How do I install Docker?" --course mlops-zoomcamp --top-k 10
```

---
## Stage 9 — Workflow Orchestration
**Folder:** [`orchestration/`](orchestration/)

The orchestration layer is a parallel track that implements the same RAG and agent
patterns using [Kestra](https://kestra.io/) instead of Python notebooks.

### Infrastructure

[`orchestration/docker-compose.yml`](orchestration/docker-compose.yml) — starts a
Kestra server (+ its Postgres backend). Requires `SECRET_GEMINI_API_KEY`,
`SECRET_TAVILY_API_KEY`, and `SECRET_GROQ_API_KEY` env vars.

```bash
cd orchestration
docker compose up -d
# UI available at http://localhost:8080  (admin@kestra.io / Admin1234!)
```

### Flows (sequential)

| Flow file | What it shows |
|-----------|---------------|
| [`1_chat_without_rag.yaml`](orchestration/flows/1_chat_without_rag.yaml) | Baseline: LLM query with no retrieved context |
| [`2_chat_with_rag.yaml`](orchestration/flows/2_chat_with_rag.yaml) | RAG via Gemini embedding store; compare with flow 1 |
| [`3_rag_with_websearch.yaml`](orchestration/flows/3_rag_with_websearch.yaml) | RAG via live Tavily web search instead of a static store |
| [`4_simple_agent.yaml`](orchestration/flows/4_simple_agent.yaml) | Single AI agent: parameterised summarisation with token tracking |
| [`5_web_research_agent.yaml`](orchestration/flows/5_web_research_agent.yaml) | Agent with web-search tool |
| [`6_multi_agent_research.yaml`](orchestration/flows/6_multi_agent_research.yaml) | Multi-agent collaboration pattern |

### Lesson notes

The [`orchestration/lessons/`](orchestration/lessons/) folder contains nine markdown
files that walk through the concepts behind each flow:

```
01-intro.md            → Module overview and learning objectives
02-context-engineering.md
03-setup.md            → How to start Kestra locally
04-ai-copilot.md
05-rag.md              → RAG theory and the two flow examples
06-agents.md
07-multi-agent.md
08-best-practices.md
09-next-steps.md
```

---
## Quick-start cheatsheet

```bash
# 1. Install dependencies
uv sync

# 2. Open this pipeline notebook
uv run jupyter notebook 00_pipeline.ipynb

# 3. Run individual focused notebooks (after Stage 2 cells are executed above)
#    minsearch-vector.ipynb    — in-memory vector search
#    rag-vector.ipynb          — RAG with vector search
#    sqlitesearch-vector.ipynb — persistent SQLite vector search
#    pgvector-vector.ipynb     — PostgreSQL vector search (needs Postgres)
#    embeddings.ipynb          — deep-dive embedding exploration

# 4. Run the pipeline from the command line (no Jupyter needed)
uv run python main.py --query "Can I still join after the start date?"
uv run python main.py --query "Can I still join after the start date?" --rag

# 5. Start the Kestra orchestration layer
cd orchestration && docker compose up -d
```

### Environment variables (`.env` file at project root)

```
OLLAMA_API_KEY=<your-key>        # used by rag-vector, sqlitesearch-vector, embeddings notebooks
OPENAI_API_KEY=<your-key>        # alternative LLM backend
# For orchestration/docker-compose.yml:
SECRET_GEMINI_API_KEY=<your-key>
SECRET_TAVILY_API_KEY=<your-key>
SECRET_GROQ_API_KEY=<your-key>
```